# Keerat: Initial SQL exploration (Utilities)

**Tables:** `Utility`, `ReportUtility` (the many-to-many junction), joined back to `UnitReport`, `OccupancyType`, `RentBand` and `LocationPoint`.
**Schema:** `main` → `own_implementation/schema.sql`, loaded with `load_3nf.py`. All 5 queries ran successfully on MySQL 8.0.
**Data:** my 8/30 snapshot (549,899 rows). *Re-run on the team zip before the slides are final; the numbers will shift slightly.*

SQL: `keerat_queries.sql`

Utilities only matter for rented units, so most queries look at tenant-occupied reports (`Occupied by non-owner`, 483,297 reports).

---

## Basic 1: Which utilities are most often included in tenants' base rent?
*JOIN, WHERE, GROUP BY, COUNT*

| utility | tenant reports | % of tenant reports |
|---|---:|---:|
| refuse_recycling | 287,294 | 59.4 |
| water_sewer | 285,131 | 59.0 |
| natural_gas | 51,906 | 10.7 |
| electricity | 31,038 | 6.4 |
| heat | 22,325 | 4.6 |
| internet / other / parking / pest_control / storage | < 1,600 each | ≤ 0.3 |

**Insight:** most landlords include water and trash (~59%). Energy (gas, electricity) is usually the tenant's own bill. That matters when comparing rents: a "$2,000" unit with utilities included isn't the same as one without.

## Basic 2: Where did each utility claim come from, the checkbox or the free-text box?
*JOIN, GROUP BY, conditional SUM*

| utility | rows | checkbox | free text | both |
|---|---:|---:|---:|---:|
| refuse_recycling | 289,621 | 289,621 | 368 | 368 |
| water_sewer | 287,480 | 287,325 | 308 | 153 |
| natural_gas | 52,447 | 52,228 | 290 | 71 |
| electricity | 31,468 | 31,468 | 0 | 0 |
| heat | 22,908 | 0 | 22,908 | 0 |
| internet | 1,522 | 0 | 1,522 | 0 |

**Insight:** heat, internet, parking, pest control and storage exist **only** because we parsed the free-text box. There's no checkbox for them, so they're probably **under-counted** (owners had to think to type them). The "both" column shows owners sometimes ticked a box *and* typed the same utility, which is why the junction table stores both source flags.

## Advanced 1: Do tenants whose rent includes more utilities report higher rent?
*CTE, LEFT JOIN, 4-table join, aggregation*

| # utilities included | tenant reports | avg rent (band midpoint) |
|---:|---:|---:|
| 0 | 174,148 | $2,817 |
| 1 | 27,562 | $2,714 |
| 2 | 200,055 | $2,376 |
| 3 | 47,511 | $2,147 |
| 4 | 22,815 | $1,836 |
| 5 | 2,257 | $1,148 |

**Insight:** the opposite of what you'd expect. The more utilities included, the **lower** the rent. Utilities aren't making rent cheaper. Both are linked to **older buildings** (see Extra), which tend to be rent-controlled with long tenancies. This is correlation, not causation.

## Advanced 2: For each utility, which supervisor districts include it most often?
*3 CTEs, window function `RANK() OVER (PARTITION BY …)`, join through LocationPoint*

| utility | #1 district (%) | #2 | #3 |
|---|---|---|---|
| water_sewer | D2 (75.7) | D1 (70.5) | D9 (67.5) |
| refuse_recycling | D2 (75.7) | D1 (68.0) | D9 (67.8) |
| heat | D2 (8.8) | D3 (7.3) | D5 (7.2) |
| natural_gas | D3 (13.6) | D9 (13.0) | D2 / D6 tie (12.5) |
| electricity | D3 (10.2) | D11 (9.5) | D9 (8.4) |

**Insight:** District 2 (Marina, Pacific Heights) leads for water, trash and heat. District 3 (Nob Hill, Chinatown, North Beach) leads for gas and electricity. Both are areas with many older apartment buildings. `RANK()` correctly gives D2 and D6 a tie for 3rd place on gas.

## Extra (for the final; we'll need 5 advanced): Are utilities included more often in older buildings?
*VIEW, CASE, conditional aggregation*

| building era | tenant reports | water | refuse | gas | electricity | heat |
|---|---:|---:|---:|---:|---:|---:|
| before 1940 | 280,814 | 67.4% | 67.7% | 11.6% | 8.6% | 7.5% |
| 1940–1979 | 116,051 | 60.1% | 60.9% | 9.2% | 4.1% | 1.0% |
| 1980 or later | 71,550 | 29.5% | 30.2% | 10.2% | 2.4% | 0.2% |

**Insight:** pre-1940 buildings include water and trash more than twice as often as post-1980 ones, and heat almost only appears there (old steam-heat buildings). This explains Advanced 1.

---

## Issues faced during query execution

1. **Zero-utility reports disappear with an inner JOIN.** 174k tenant reports have no `ReportUtility` rows. Advanced 1 needs a `LEFT JOIN` + `COUNT(ru.utility_id)` (not `COUNT(*)`), or the "0 utilities" group silently vanishes.
2. **Rent is a band, not a number.** I used the band midpoint as an estimate, and excluded "no rent paid" and the open-ended "$7000+" band.
3. **Heat has no checkbox.** It's only found by keyword-matching free text, so its percentages are a lower bound.
4. **Keyword matching is imperfect.** The loader maps any text containing "water" (e.g. "hot water") to `water_sewer`.
5. **Performance.** Each query takes a few seconds on 550k reports + 689k utility rows. `EXPLAIN` shows the joins already use the `ReportUtility` primary key and the indexes MySQL creates automatically for each foreign key. For the final, we can test a composite index such as `UnitReport(occupancy_type_id, rent_band_id)` with `EXPLAIN ANALYZE`, since almost every query filters on tenant-occupied and then looks up rent.
